Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Routes — agent

 **Three endpoints:**
 - `POST /agent/chat` — start or continue a conversation
 - `GET /agent/stream/{session_id}` — stream graph events via SSE
 - `POST /agent/approve` — resume a suspended graph after human approval

 Full memory cycle on every request: **load LTM → run graph → save LTM**

In [ ]:
import json
import asyncio
import logging
from fastapi import APIRouter, Depends, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from sqlalchemy.ext.asyncio import AsyncSession
from app.core.database import get_db
from app.agent.graph import get_agent
from app.agent.memory.short_term import get_thread_config
from app.agent.memory.long_term import LongTermMemory
from app.core.config import get_settings
from app.agent.state import DEFAULT_MAX_ITERATIONS
from langgraph.types import Command
from langchain_groq import ChatGroq

logger = logging.getLogger(__name__)
router = APIRouter(prefix="/agent", tags=["Agent"])

settings = get_settings()
LTM = LongTermMemory()
_ltm_llm: ChatGroq | None = None

 ## `_get_ltm_llm`

 Lazy singleton — the instance is created on first call, reused after.
 Bug fix: the original was missing the `return` statement,
 so every call created a fresh `ChatGroq` instance.

In [ ]:
def _get_ltm_llm() -> ChatGroq:
    global _ltm_llm
    if _ltm_llm is None:
        _ltm_llm = ChatGroq(model=settings.llm_clarifier, temperature=0)
    return _ltm_llm

 ## Request models

In [ ]:
class AgentRequest(BaseModel):
    message: str
    session_id: str
    user_id: str = "anonymous"
    max_iterations: int = DEFAULT_MAX_ITERATIONS


class HumanApprovalRequest(BaseModel):
    session_id: str
    user_id: str
    approved: bool
    feedback: str = ""

 ## `POST /agent/chat`

 Loads LTM context before invoking the graph, injects it into the initial state.
 After the graph completes, extracts new facts and persists them.

 `workflow_trace` and `tool_call_history` are reset to empty lists on each call —
 otherwise state from a previous invocation on the same thread could bleed in.

In [ ]:
@router.post("/chat")
async def chat(req: AgentRequest, db: AsyncSession = Depends(get_db)):
    try:
        agent = get_agent()
        config = get_thread_config(req.session_id, req.user_id)
        user_context = await LTM.load_user_context(req.user_id, db)

        result = await agent.ainvoke(
            {
                "messages": [{"role": "user", "content": req.message}],
                "session_id": req.session_id,
                "user_id": req.user_id,
                "user_context": user_context,
                "max_iterations": req.max_iterations,
                "workflow_trace": [],
                "tool_call_history": [],
            },
            config=config,
        )

        new_facts = await LTM.extract_facts_from_session(result["messages"], _get_ltm_llm())
        if new_facts:
            await LTM.update_user_context(req.user_id, new_facts, db)

        return {
            "response":                result.get("final_response", ""),
            "workflow_trace":          result.get("workflow_trace", []),
            "selected_tools":          result.get("selected_tools", []),
            "iterations":              result.get("iterations", 0),
            "blocked":                 result.get("blocked", False),
            "requires_human_approval": result.get("requires_human_approval", False),
            "hallucination_warning":   result.get("hallucination_warning", False),
        }
    except Exception as e:
        logger.error("chat_error: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Internal server error")

 ## `GET /agent/stream/{session_id}` — SSE

 Passes `None` as graph input — means "resume from the last checkpoint for this thread".
 Emits four event types: `node_start`, `node_end`, `tool_start`, `tool_end`.
 The frontend WorkflowPanel listens to these to animate nodes in real time.

 `await asyncio.sleep(0)` yields control back to the event loop on each iteration —
 lets other requests process while the stream is running.

 Two required headers:
 - `Cache-Control: no-cache` — prevents proxies from buffering the stream
 - `X-Accel-Buffering: no` — disables Nginx buffering, essential behind a reverse proxy

In [ ]:
@router.get("/stream/{session_id}")
async def stream_workflow(session_id: str, user_id: str = "anonymous"):
    try:
        agent = get_agent()
        config = get_thread_config(session_id, user_id)
    except Exception as e:
        logger.error("stream_init_error: %s", e, exc_info=True)

        async def error_generator():
            yield f"data: {json.dumps({'type': 'error', 'message': str(e), 'code': 'initialization_error'})}\n\n"

        return StreamingResponse(
            error_generator(),
            media_type="text/event-stream",
            headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
        )

    async def event_generator():
        try:
            async for event in agent.astream_events(None, config=config, version="v2"):
                event_type = event.get("event")
                node_name  = event.get("name", "")

                if event_type == "on_chain_start":
                    yield f"data: {json.dumps({'type': 'node_start', 'node': node_name})}\n\n"
                elif event_type == "on_chain_end":
                    yield f"data: {json.dumps({'type': 'node_end', 'node': node_name})}\n\n"
                elif event_type == "on_tool_start":
                    yield f"data: {json.dumps({'type': 'tool_start', 'tool': node_name})}\n\n"
                elif event_type == "on_tool_end":
                    yield f"data: {json.dumps({'type': 'tool_end', 'tool': node_name})}\n\n"

                await asyncio.sleep(0)
        except Exception as e:
            logger.error("stream_execution_error: %s", e, exc_info=True)
            yield f"data: {json.dumps({'type': 'error', 'message': str(e), 'code': 'execution_error'})}\n\n"

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )

 ## `POST /agent/approve`

 `Command(resume=...)` is the LangGraph primitive for injecting a value into
 a suspended `interrupt()`. The graph loads from the checkpointer using the same
 `thread_id`, resumes from the exact line where `interrupt()` was called,
 and the dict becomes the return value of `interrupt()` in `human_loop.py`.

 Fact extraction runs again after resuming — the approval flow may have produced
 new conversational content worth remembering.

In [ ]:
@router.post("/approve")
async def approve_action(req: HumanApprovalRequest, db: AsyncSession = Depends(get_db)):
    try:
        agent = get_agent()
        config = get_thread_config(req.session_id, req.user_id)

        result = await agent.ainvoke(
            Command(resume={"approved": req.approved, "feedback": req.feedback}),
            config=config,
        )

        if "messages" in result:
            new_facts = await LTM.extract_facts_from_session(result["messages"], _get_ltm_llm())
            if new_facts:
                await LTM.update_user_context(req.user_id, new_facts, db)

        return {
            "response": result.get("final_response", ""),
            "approved": req.approved,
        }
    except Exception as e:
        logger.error("approval_error: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Internal server error")